# Quantum fleet assignment with QAOA

**Team OOPs · VTU-Qiskit Fall Fest 2026 · VTU-QFF-HC-0107**

A factory in Pune has open loads, and transporters have trucks with route legs and quotes. Which truck should carry each load?

This notebook walks through the exact quantum formulation behind the **Quantum Fleet Optimizer** page of LoadSetu (`/factory/optimizer`):

1. the real problem, taken from the app's seeded data,
2. the classical exact answer,
3. the QUBO, with the penalty weights justified,
4. the Ising Hamiltonian,
5. QAOA on a noiseless simulator and on a simulator with IBM Brisbane's noise model,
6. what the results do and do not show.

Run from the `quantum/` folder with the venv active (`pip install -r ../requirements.txt`).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # notebooks/ -> quantum/

import numpy as np
import matplotlib.pyplot as plt
from itertools import product

from qlogistics.demo import DEMO_OPTIONS, LOADS
from qlogistics.qubo import build_qubo, to_ising, brute_force, bits_to_x
from qlogistics.qaoa import solve, bind
from qiskit.circuit.library import qaoa_ansatz
import qiskit
print("Qiskit", qiskit.__version__)

## 1. The instance

The matching engine (`backend/src/services/matchingEngine.js`) has already removed every truck that is the wrong type, too small, or has no available route leg covering the trip. What is left are the **eligible pairs**, each with the transporter's quote. One pair becomes one binary decision variable, and later one qubit.

In [ ]:
for i, o in enumerate(DEMO_OPTIONS):
    print(f"x{i}: {LOADS[o.load_id]:<45} truck {o.truck_id}   Rs {o.cost:>9,.0f}")

## 2. Classical answer

Rules: each load gets at most one truck, each truck carries at most one load. Objective: **assign as many loads as possible, then minimise total cost**. That is the objective of the app's exact branch-and-bound solver (`assignmentOptimizer.js`). With 9 variables we can simply try all $2^9 = 512$ subsets.

In [ ]:
def classical(options):
    best = None
    for pick in product([0, 1], repeat=len(options)):
        chosen = [o for o, b in zip(options, pick) if b]
        if len({o.load_id for o in chosen}) < len(chosen) or len({o.truck_id for o in chosen}) < len(chosen):
            continue  # a load or a truck used twice
        key = (-len(chosen), sum(o.cost for o in chosen))
        if best is None or key < best[0]:
            best = (key, chosen)
    return best[1]

best = classical(DEMO_OPTIONS)
for o in best:
    print(f"{LOADS[o.load_id]:<45} -> {o.truck_id}  Rs {o.cost:,.0f}")
print(f"\n{len(best)} loads, total Rs {sum(o.cost for o in best):,.0f}")

## 3. The QUBO

A QUBO (quadratic unconstrained binary optimisation) has no constraints, so the rules go into the energy as penalties:

$$E(x) = \sum_i (c_i - A)\,x_i \;+\; P \sum_{i<j\ \text{conflict}} x_i x_j$$

* $c_i = \text{cost}_i / \max\text{cost} \in [0, 1]$, so the scale does not depend on rupee amounts.
* $A = (\text{number of loads}) + 1$ is a **reward** for each assignment. All the normalised costs together are at most one per load, so they sum to less than $A$. One more assigned load therefore always beats any cost saving: count first, cost second.
* Two options **conflict** if they share a load or a truck. $P = 2A$, so adding a clashing option changes the energy by $c_j - A + P > 0$. A clash never pays, and every minimum of $E$ is a valid assignment.

Both claims are checked against brute force on 40 random instances in `tests/test_qubo.py`.

In [ ]:
q = build_qubo(DEMO_OPTIONS)
print(f"qubits: {q.num_qubits}   reward A = {q.reward}   penalty P = {q.penalty}")
print(f"conflicting pairs: {len(q.quadratic)}")
print("linear terms h_i:", np.round(q.linear, 3))

opt_x, opt_e = brute_force(q)
print("\nQUBO minimum:", [f"{o.load_id}->{o.truck_id}" for o in q.decode(opt_x)], f"E = {opt_e:.4f}")
print("same as classical:", set(q.decode(opt_x)) == set(best))

The ten lowest energies. Every low-energy state is feasible, and states that assign fewer loads sit clearly above the optimum:

In [ ]:
n = q.num_qubits
states = [np.array([(k >> i) & 1 for i in range(n)]) for k in range(2 ** n)]
ranked = sorted(states, key=q.energy)[:10]
for x in ranked:
    picked = q.decode(x)
    print(f"E = {q.energy(x):8.4f}  feasible={q.is_feasible(x)!s:5}  loads={len(picked)}  Rs {sum(o.cost for o in picked):>7,.0f}")

## 4. Ising Hamiltonian

Quantum hardware measures $Z$, with eigenvalues $\pm 1$. Substituting $x_i = (1 - Z_i)/2$ turns the QUBO into a cost Hamiltonian

$$H = \sum_i h_i Z_i + \sum_{i<j} J_{ij} Z_i Z_j + \text{offset},$$

whose ground state is the optimal assignment. In Qiskit bitstrings, qubit 0 is the **rightmost** character.

In [ ]:
H, offset = to_ising(q)
print(f"{len(H)} Pauli terms, offset {offset:.3f}")
print(H)

# Check: <x|H|x> + offset equals the QUBO energy for every basis state.
diag = H.to_matrix(sparse=True).diagonal().real
print("\nmatches QUBO on all 512 states:", all(abs(diag[k] + offset - q.energy(states[k])) < 1e-9 for k in range(2 ** n)))

## 5. QAOA

QAOA starts in an equal superposition and applies $p$ layers of $e^{-i\gamma_k H}$ (cost) followed by $e^{-i\beta_k \sum X}$ (mixer). A classical optimiser (COBYLA) tunes $\gamma, \beta$ to lower $\langle H \rangle$. Then we sample the circuit and keep the best **valid** bitstring.

One layer of the circuit, for a picture of its shape:

In [ ]:
qaoa_ansatz(H / np.max(np.abs(H.coeffs)), reps=1).decompose().draw("text", fold=110)

### 5a. Noiseless simulator

$p = 2$ layers, 2,048 shots. Training uses an exact NumPy statevector of the diagonal Hamiltonian (checked against the Qiskit circuit in the tests), because rebuilding the Qiskit circuit every optimiser step took minutes at 12 qubits.

In [ ]:
ideal = solve(q, mode="ideal", reps=2, shots=2048, seed=7)

def summary(r):
    picked = q.decode(r.x) if r.x is not None else []
    return {
        "backend": r.backend,
        "best valid sample": [f"{o.load_id}->{o.truck_id}" for o in picked],
        "cost (Rs)": sum(o.cost for o in picked),
        "matches optimum": r.energy is not None and abs(r.energy - r.optimal_energy) < 1e-9,
        "P(optimal bitstring)": round(r.probability_of_optimal, 4),
        "valid samples": round(r.feasible_shot_fraction, 3),
        "transpiled depth": r.circuit["transpiledDepth"],
        "two-qubit gates": r.circuit["twoQubitGates"],
        "runtime (s)": round(r.runtime_ms / 1000, 1),
    }

summary(ideal)

### 5b. Simulator with IBM Brisbane's noise model

Same trained parameters, but the circuit is routed onto Brisbane's heavy-hex layout and its native gates, and the simulator applies that device's measured error rates. Routing adds SWAPs, so depth grows a lot.

In [ ]:
noisy = solve(q, mode="noisy", reps=2, shots=2048, seed=7)
summary(noisy)

In [ ]:
def top(counts, k=12):
    return sorted(counts.items(), key=lambda kv: -kv[1])[:k]

opt_key = "".join(str(b) for b in reversed(opt_x))
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, r, title in [(axes[0], ideal, "Noiseless"), (axes[1], noisy, "IBM Brisbane noise model")]:
    rows = top(r.counts)
    colors = ["#2f9e77" if b == opt_key else ("#e4a444" if q.is_feasible(bits_to_x(b)) else "#b0b7b5") for b, _ in rows]
    ax.bar(range(len(rows)), [c / r.shots for _, c in rows], color=colors)
    ax.set_xticks(range(len(rows)), [b for b, _ in rows], rotation=70, fontsize=8, family="monospace")
    ax.set_title(f"{title}: depth {r.circuit['transpiledDepth']}, valid {r.feasible_shot_fraction:.0%}")
axes[0].set_ylabel("probability")
fig.suptitle("Most frequent bitstrings (green = optimal, amber = valid, grey = breaks a rule)")
plt.tight_layout()

## 6. What this shows, honestly

* **The formulation is right.** The QUBO minimum equals the classical optimum (Rs 59,500, all 4 loads), and both QAOA runs find it among their samples.
* **QAOA is not faster here.** The exact classical solver takes under a millisecond. Brute-forcing 9 qubits is trivial, and the probability of measuring the optimal bitstring directly is only a few percent. A quantum advantage for this problem would need far larger instances and better hardware than exist today.
* **Noise is visible.** On the Brisbane noise model the circuit is more than 10× deeper after routing, and the share of valid samples drops. That is the real engineering limit.
* **Scale limit.** Simulation cost doubles per qubit, so the service caps a run at 12 qubits (10 for noisy) by keeping the cheapest options per load, and says so in the UI when it does.

To run the same circuit on real IBM Quantum hardware, see `scripts/run_on_ibm.py`.